# Step 7: Statistical Analysis & Hypothesis Testing
Rigorous testing:
1. Independent Two-Sample T-Test (Delivery Status vs Review Score)
2. One-Way ANOVA (Product Category vs Order Value)
3. Chi-Square Test of Independence (Payment Method vs Order Status)


In [1]:
import os
import pandas as pd
from scipy import stats
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

load_dotenv()

MYSQL_HOST = os.getenv("MYSQL_HOST", "localhost")
MYSQL_PORT = os.getenv("MYSQL_PORT", "3306")
MYSQL_USER = os.getenv("MYSQL_USER", "root")
MYSQL_PASSWORD = os.getenv("MYSQL_PASSWORD", "root")
MYSQL_DB = os.getenv("MYSQL_DB", "cart2insights_db")

mysql_uri = f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DB}"
engine = create_engine(mysql_uri)
print(f"Connected to MySQL database: {MYSQL_DB} on {MYSQL_HOST}:{MYSQL_PORT}")

orders = pd.read_sql(text("SELECT * FROM orders"), engine)
reviews = pd.read_sql(text("SELECT * FROM order_reviews"), engine)

orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])
orders['order_estimated_delivery_date'] = pd.to_datetime(orders['order_estimated_delivery_date'])
orders['delivery_delay'] = (orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']).dt.days
orders['is_delayed'] = orders['delivery_delay'] > 0

df = orders.merge(reviews, on='order_id')
delayed = df[df['is_delayed'] == True]['review_score'].dropna()
ontime = df[df['is_delayed'] == False]['review_score'].dropna()

t_stat, p_val = stats.ttest_ind(delayed, ontime, equal_var=False)
print(f"Independent T-Test Results: T-statistic={t_stat:.4f}, p-value={p_val:.4e}", flush=True)
if p_val < 0.05:
    print("Conclusion: Reject H0. Delayed orders receive significantly lower review scores.", flush=True)
else:
    print("Conclusion: Fail to reject H0.", flush=True)


Connected to MySQL database: cart2insights_db on localhost:3306
Independent T-Test Results: T-statistic=-96.8492, p-value=0.0000e+00
Conclusion: Reject H0. Delayed orders receive significantly lower review scores.
